In [1]:
import sys
import pandas as pd
import arch
import yfinance as yf

print("Python executable:", sys.executable)
print("Environment ready")

Python executable: /Users/lohith/Documents/GitHub/FTSE100-Volatility-Garch/.venv/bin/python
Environment ready


# FTSE 100: Data acquisition and preparation

## Objective
Acquire daily FTSE 100 price-index observations and inspect date coverage,
missing values, duplicate dates and OHLC consistency.

## Planned sample
January 2000 to August 2026.

A short December 1999 buffer is downloaded so that the first January 2000
return can be calculated using the preceding trading day's closing price.

## Source
Yahoo Finance, accessed through yfinance. Ticker: ^FTSE.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json

import pandas as pd
import yfinance as yf
from IPython.display import display

# Locate the project whether VS Code runs this notebook
# from the project root or the notebooks folder.
current_folder = Path.cwd()

project_root = next(
    (
        folder
        for folder in [current_folder, *current_folder.parents]
        if (folder / "requirements.txt").is_file()
        and (folder / "notebooks").is_dir()
        and (folder / "data").is_dir()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("Could not locate the FTSE project folder.")

raw_folder = project_root / "data" / "raw"

print("Project folder:", project_root)
print("Raw data folder:", raw_folder)

Project folder: /Users/lohith/Documents/GitHub/FTSE100-Volatility-Garch
Raw data folder: /Users/lohith/Documents/GitHub/FTSE100-Volatility-Garch/data/raw


In [3]:
ticker = "^FTSE"
download_start = "1999-12-01"
download_end = "2026-09-01"

ftse_raw = yf.download(
    tickers=ticker,
    start=download_start,
    end=download_end,
    interval="1d",
    auto_adjust=False,
    back_adjust=False,
    repair=False,
    keepna=True,
    actions=False,
    multi_level_index=False,
    progress=False,
    threads=False,
)

if ftse_raw is None or ftse_raw.empty:
    raise RuntimeError(
        "No observations were downloaded. Check the download message above."
    )

retrieved_at = datetime.now(timezone.utc).isoformat()

print(f"Downloaded {len(ftse_raw):,} rows.")

Downloaded 6,808 rows.


In [4]:
# A timestamp prevents a later download from overwriting this snapshot.
snapshot = datetime.fromisoformat(retrieved_at).strftime("%Y%m%dT%H%M%S%fZ")

raw_path = raw_folder / f"ftse100_daily_{snapshot}.csv"
metadata_path = raw_folder / f"ftse100_daily_{snapshot}_metadata.json"

ftse_raw.to_csv(raw_path, index_label="Date", mode="x")

metadata = {
    "ticker": ticker,
    "source": "Yahoo Finance via yfinance",
    "retrieved_at_utc": retrieved_at,
    "requested_start_inclusive": download_start,
    "requested_end_exclusive": download_end,
    "intended_analysis_start": "2000-01-01",
    "intended_analysis_end": "2026-08-31",
    "interval": "1d",
    "auto_adjust": False,
    "back_adjust": False,
    "repair": False,
    "keepna": True,
    "rows": len(ftse_raw),
    "first_observation": str(ftse_raw.index.min()),
    "last_observation": str(ftse_raw.index.max()),
    "columns": ftse_raw.columns.tolist(),
    "yfinance_version": yf.__version__,
    "pandas_version": pd.__version__,
    "raw_file": raw_path.name,
}

with metadata_path.open("x", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved:", raw_path.name)
print("Saved:", metadata_path.name)

Saved: ftse100_daily_20260919T060218484765Z.csv
Saved: ftse100_daily_20260919T060218484765Z_metadata.json


In [5]:
print("First observation:", ftse_raw.index.min())
print("Last observation:", ftse_raw.index.max())
print("Total rows:", len(ftse_raw))
print("Duplicate dates:", ftse_raw.index.duplicated().sum())
print("Dates in ascending order:", ftse_raw.index.is_monotonic_increasing)
print("Weekend observations:", (ftse_raw.index.dayofweek >= 5).sum())

print("\nFirst five rows:")
display(ftse_raw.head())

print("\nLast five rows:")
display(ftse_raw.tail())

print("\nColumn types and missing values:")
display(
    pd.DataFrame({
        "dtype": ftse_raw.dtypes.astype(str),
        "missing_count": ftse_raw.isna().sum(),
        "missing_percent": 100 * ftse_raw.isna().mean(),
    })
)

print("\nObservations by year:")
display(ftse_raw.groupby(ftse_raw.index.year).size().rename("observations"))

First observation: 1999-12-01 00:00:00
Last observation: 2026-08-28 00:00:00
Total rows: 6808
Duplicate dates: 0
Dates in ascending order: True
Weekend observations: 0

First five rows:


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
1999-12-01,6646.000000,6646.000000,6655.799805,6575.299805,6597.200195,642023000
1999-12-02,6653.700195,6653.700195,6694.200195,6623.000000,6646.000000,946100000
1999-12-03,6742.200195,6742.200195,6772.100098,6652.299805,6653.700195,954355000
1999-12-06,6694.000000,6694.000000,6769.399902,6657.100098,6742.200195,640360000
1999-12-07,6660.899902,6660.899902,6728.899902,6659.700195,6694.000000,729797000



Last five rows:


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-08-24,10854.299805,10854.299805,10864.799805,10799.000000,10816.299805,664816800
2026-08-25,10886.200195,10886.200195,10887.000000,10825.799805,10854.599609,618073600
2026-08-26,10878.099609,10878.099609,10920.500000,10855.599609,10886.200195,587582300
2026-08-27,10792.500000,10792.500000,10878.500000,10772.599609,10878.200195,605676800
2026-08-28,10824.299805,10824.299805,10845.000000,10792.400391,10792.500000,968083200



Column types and missing values:


,dtype,missing_count,missing_percent
Adj Close,float64,54,0.793184
Close,float64,54,0.793184
High,float64,54,0.793184
Low,float64,54,0.793184
Open,float64,54,0.793184
Volume,int64,0,0.000000



Observations by year:


Date
1999     23
2000    260
2001    261
2002    261
2003    261
2004    262
2005    260
2006    252
2007    253
2008    254
2009    253
2010    253
2011    252
2012    251
2013    253
2014    253
2015    253
2016    253
2017    252
2018    253
2019    253
2020    254
2021    253
2022    250
2023    251
2024    254
2025    253
2026    167
Name: observations, dtype: int64

In [6]:
price_columns = ["Open", "High", "Low", "Close"]

missing_columns = [
    column for column in price_columns
    if column not in ftse_raw.columns
]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

prices = ftse_raw[price_columns]
complete_ohlc = prices.notna().all(axis=1)

checks = pd.DataFrame(index=ftse_raw.index)

checks["missing_OHLC"] = prices.isna().any(axis=1)
checks["nonpositive_price"] = prices.le(0).any(axis=1)
checks["high_below_low"] = ftse_raw["High"] < ftse_raw["Low"]

checks["open_outside_range"] = complete_ohlc & (
    (ftse_raw["Open"] < ftse_raw["Low"])
    | (ftse_raw["Open"] > ftse_raw["High"])
)

checks["close_outside_range"] = complete_ohlc & (
    (ftse_raw["Close"] < ftse_raw["Low"])
    | (ftse_raw["Close"] > ftse_raw["High"])
)

checks["zero_range"] = ftse_raw["High"] == ftse_raw["Low"]

print("Number of rows flagged by each check:")
display(checks.sum().rename("flagged_rows"))

flagged_rows = checks.any(axis=1)

print("\nFirst 20 flagged observations:")
display(ftse_raw.loc[flagged_rows].head(20))

Number of rows flagged by each check:


missing_OHLC           54
nonpositive_price       0
high_below_low          0
open_outside_range      0
close_outside_range     0
zero_range              0
Name: flagged_rows, dtype: int64


First 20 flagged observations:


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
1999-12-27,NaN,NaN,NaN,NaN,NaN,0
1999-12-28,NaN,NaN,NaN,NaN,NaN,0
1999-12-31,NaN,NaN,NaN,NaN,NaN,0
2000-01-03,NaN,NaN,NaN,NaN,NaN,0
2000-04-21,NaN,NaN,NaN,NaN,NaN,0
2000-04-24,NaN,NaN,NaN,NaN,NaN,0
2000-05-01,NaN,NaN,NaN,NaN,NaN,0
2000-05-29,NaN,NaN,NaN,NaN,NaN,0
2000-08-28,NaN,NaN,NaN,NaN,NaN,0


In [7]:
price_columns = ["Open", "High", "Low", "Close"]

any_price_missing = ftse_raw[price_columns].isna().any(axis=1)
all_prices_missing = ftse_raw[price_columns].isna().all(axis=1)

print("Rows with all four prices missing:", all_prices_missing.sum())
print(
    "Rows with only some prices missing:",
    (any_price_missing & ~all_prices_missing).sum(),
)

print("\nAll dates with missing prices:")
print(
    ftse_raw.loc[any_price_missing, price_columns + ["Volume"]]
    .to_string()
)

annual_counts = (
    ftse_raw.groupby(ftse_raw.index.year)
    .agg(
        total_rows=("Close", "size"),
        valid_closes=("Close", "count"),
    )
)

annual_counts["missing_closes"] = (
    annual_counts["total_rows"] - annual_counts["valid_closes"]
)

print("\nAnnual coverage:")
display(annual_counts)

Rows with all four prices missing: 54
Rows with only some prices missing: 0

All dates with missing prices:
            Open  High  Low  Close  Volume
Date                                      
1999-12-27   NaN   NaN  NaN    NaN       0
1999-12-28   NaN   NaN  NaN    NaN       0
1999-12-31   NaN   NaN  NaN    NaN       0
2000-01-03   NaN   NaN  NaN    NaN       0
2000-04-21   NaN   NaN  NaN    NaN       0
2000-04-24   NaN   NaN  NaN    NaN       0
2000-05-01   NaN   NaN  NaN    NaN       0
2000-05-29   NaN   NaN  NaN    NaN       0
2000-08-28   NaN   NaN  NaN    NaN       0
2000-12-25   NaN   NaN  NaN    NaN       0
2000-12-26   NaN   NaN  NaN    NaN       0
2001-01-01   NaN   NaN  NaN    NaN       0
2001-04-13   NaN   NaN  NaN    NaN       0
2001-04-16   NaN   NaN  NaN    NaN       0
2001-05-07   NaN   NaN  NaN    NaN       0
2001-05-28   NaN   NaN  NaN    NaN       0
2001-08-27   NaN   NaN  NaN    NaN       0
2001-12-25   NaN   NaN  NaN    NaN       0
2001-12-26   NaN   NaN  NaN    N

,total_rows,valid_closes,missing_closes
Date,,,
1999,23,20,3
2000,260,252,8
2001,261,253,8
2002,261,252,9
2003,261,253,8
2004,262,254,8
2005,260,252,8
2006,252,252,0
2007,253,253,0


In [8]:
# Request a short window around the suspicious missing observation.
ftse_retry = yf.download(
    tickers="^FTSE",
    start="2020-12-18",
    end="2020-12-25",
    interval="1d",
    auto_adjust=False,
    back_adjust=False,
    repair=False,
    keepna=True,
    actions=False,
    multi_level_index=False,
    progress=False,
    threads=False,
)

print("Original download:")
display(
    ftse_raw.loc[
        "2020-12-18":"2020-12-24",
        ["Open", "High", "Low", "Close", "Volume"],
    ]
)

print("\nSeparate short-window download:")
display(ftse_retry)

Original download:


,Open,High,Low,Close,Volume
Date,,,,,
2020-12-18,6551.100098,6604.500000,6518.299805,6529.200195,1200513100
2020-12-21,6529.200195,6529.200195,6315.299805,6416.299805,1118314800
2020-12-22,NaN,NaN,NaN,NaN,0
2020-12-23,6453.200195,6497.299805,6431.200195,6495.799805,807695400
2020-12-24,6495.799805,6530.299805,6479.600098,6502.100098,477470700



Separate short-window download:


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2020-12-18,6529.200195,6529.200195,6604.500000,6518.299805,6551.100098,1200513100
2020-12-21,6416.299805,6416.299805,6529.200195,6315.299805,6529.200195,1118314800
2020-12-22,NaN,NaN,NaN,NaN,NaN,0
2020-12-23,6495.799805,6495.799805,6497.299805,6431.200195,6453.200195,807695400
2020-12-24,6502.100098,6502.100098,6530.299805,6479.600098,6495.799805,477470700


In [14]:
from urllib.parse import urlunsplit

source_url = urlunsplit((
    "https",
    "www.investing.com",
    "/indices/uk-100-historical-data",
    "",
    "",
))

corrections = pd.DataFrame({
    "Date": pd.to_datetime(["2020-12-22"]),
    "Open": [6416.32],
    "High": [6457.47],
    "Low": [6382.61],
    "Close": [6453.16],
    "source": ["Investing.com"],
    "source_url": [source_url],
    "evidence_file": ["investing_ftse100_20201218_20201224.png"],
    "reason": [
        "Yahoo OHLC missing in both full-sample and short-window downloads; "
        "replacement transcribed from historical-data screenshot."
    ],
}).set_index("Date")

corrections_folder = project_root / "data" / "corrections"
corrections_folder.mkdir(exist_ok=True)

corrections.to_csv(
    corrections_folder / "ftse100_price_corrections.csv",
    index_label="Date",
)

display(corrections[["Open", "High", "Low", "Close", "source"]])

,Open,High,Low,Close,source
Date,,,,,
2020-12-22,6416.32,6457.47,6382.61,6453.16,Investing.com


In [15]:
# Preserve ftse_raw unchanged.
ftse_working = ftse_raw.copy()

ftse_working["price_source"] = "Yahoo Finance"
ftse_working["price_corrected"] = False

price_columns = ["Open", "High", "Low", "Close"]
correction_date = pd.Timestamp("2020-12-22")

# Stop if the original observation no longer has the expected missing prices.
assert ftse_working.loc[correction_date, price_columns].isna().all()

ftse_working.loc[correction_date, price_columns] = (
    corrections.loc[correction_date, price_columns].astype(float)
)

ftse_working.loc[correction_date, "price_source"] = "Investing.com"
ftse_working.loc[correction_date, "price_corrected"] = True

# Yahoo's zero volume accompanied missing prices.
# The screenshot's volume is rounded, so leave volume unavailable.
ftse_working.loc[correction_date, "Volume"] = float("nan")

display(
    ftse_working.loc[
        "2020-12-21":"2020-12-23",
        price_columns + ["price_source", "price_corrected"],
    ]
)

print(
    "Rows still missing any OHLC:",
    ftse_working[price_columns].isna().any(axis=1).sum(),
)

,Open,High,Low,Close,price_source,price_corrected
Date,,,,,,
2020-12-21,6529.200195,6529.200195,6315.299805,6416.299805,Yahoo Finance,False
2020-12-22,6416.320000,6457.470000,6382.610000,6453.160000,Investing.com,True
2020-12-23,6453.200195,6497.299805,6431.200195,6495.799805,Yahoo Finance,False


Rows still missing any OHLC: 53


### Missing observation: 22 December 2020

Yahoo Finance returned missing OHLC prices for this date in both the
full-sample download and a separate short-window request.

The missing OHLC values were replaced in a working copy using
Investing.com's FTSE 100 historical-data table. Neighbouring closing
prices differed from Yahoo by no more than 0.05 index points across
the four comparison dates.

The original download was preserved. The replacement values, source
and reason are recorded in data/corrections/ftse100_price_corrections.csv.
Volume and adjusted close remain unavailable for the corrected date.

In [21]:
reference_closes = pd.Series(
    [5351.53, 5391.14, 5297.28],
    index=pd.to_datetime([
        "2012-05-25",
        "2012-05-29",
        "2012-05-30",
    ]),
    name="Investing_Close",
)

comparison = pd.concat(
    [
        ftse_raw["Close"].reindex(
            reference_closes.index
        ).rename("Yahoo_Close"),
        reference_closes,
    ],
    axis=1,
)

comparison["absolute_difference"] = (
    comparison["Yahoo_Close"] - comparison["Investing_Close"]
).abs()

display(comparison)

# Flag discrepancies beyond a small rounding tolerance.
assert comparison["Yahoo_Close"].notna().all()
assert comparison["absolute_difference"].le(0.10).all(), (
    "Neighbouring closes differ by more than 0.10 index points. "
    "Review the comparison before applying the correction."
)

,Yahoo_Close,Investing_Close,absolute_difference
2012-05-25,5351.500000,5351.53,0.030000
2012-05-29,5391.100098,5391.14,0.039902
2012-05-30,5297.299805,5297.28,0.019805


In [22]:
correction_date = pd.Timestamp("2012-05-28")

# Add or replace this entry, making reruns safe.
corrections.loc[correction_date] = {
    "Open": 5351.53,
    "High": 5413.83,
    "Low": 5341.34,
    "Close": 5356.34,
    "source": "Investing.com",
    "source_url": source_url,
    "evidence_file": "investing_ftse100_20120525_20120530.png",
    "reason": (
        "Date absent from Yahoo download but identified as an XLON "
        "session; OHLC transcribed from historical-data screenshot."
    ),
}

corrections = corrections.sort_index()
price_columns = ["Open", "High", "Low", "Close"]

# Protect any existing valid Yahoo prices from being overwritten.
existing_correction_dates = corrections.index.intersection(ftse_raw.index)

assert (
    ftse_raw.loc[existing_correction_dates, price_columns]
    .isna().all().all()
)

# Include the absent 2012 date and retain all original rows.
combined_dates = ftse_raw.index.union(corrections.index).sort_values()
ftse_working = ftse_raw.reindex(combined_dates).copy()

ftse_working["price_source"] = "Yahoo Finance"
ftse_working["price_corrected"] = False

# Apply BOTH the 2012 and 2020 corrections.
ftse_working.loc[corrections.index, price_columns] = (
    corrections[price_columns].to_numpy(dtype=float)
)

ftse_working.loc[corrections.index, "price_source"] = "Investing.com"
ftse_working.loc[corrections.index, "price_corrected"] = True
ftse_working.loc[corrections.index, "Volume"] = float("nan")

# Save the updated two-row audit table.
corrections.to_csv(
    corrections_folder / "ftse100_price_corrections.csv",
    index_label="Date",
)

display(
    ftse_working.loc[
        corrections.index,
        price_columns + ["price_source", "price_corrected"],
    ]
)

,Open,High,Low,Close,price_source,price_corrected
Date,,,,,,
2012-05-28,5351.53,5413.83,5341.34,5356.34,Investing.com,True
2020-12-22,6416.32,6457.47,6382.61,6453.16,Investing.com,True


In [23]:
import exchange_calendars as xcals

calendar_start = "1999-12-01"
calendar_end = "2026-08-31"

# Build a calendar extending beyond both sample boundaries.
lse = xcals.get_calendar(
    "XLON",
    start="1999-11-01",
    end="2026-09-07",
)

# Still request only sessions within our original sample period.
expected_sessions = lse.sessions_in_range(
    calendar_start,
    calendar_end,
).tz_localize(None)

observed_dates = ftse_working.index.tz_localize(None)

price_columns = ["Open", "High", "Low", "Close"]

complete_prices = ftse_working[price_columns].notna().all(axis=1)
empty_prices = ftse_working[price_columns].isna().all(axis=1)
partial_prices = ~complete_prices & ~empty_prices

valid_price_dates = observed_dates[complete_prices.to_numpy()]
empty_price_dates = observed_dates[empty_prices.to_numpy()]

# Missing-price rows that the calendar identifies as non-trading dates.
removable_dates = empty_price_dates.difference(expected_sessions)

# Expected sessions without complete prices, including dates absent entirely.
unresolved_sessions = expected_sessions.difference(valid_price_dates)

# Complete prices appearing on dates the calendar says were closed.
unexpected_price_dates = valid_price_dates.difference(expected_sessions)

print("Empty rows on non-trading dates:", len(removable_dates))
print("Expected sessions without complete OHLC:", len(unresolved_sessions))
print("Complete-price rows outside the calendar:", len(unexpected_price_dates))
print("Partially missing OHLC rows:", partial_prices.sum())

if len(unresolved_sessions):
    print("\nSessions requiring investigation:")
    print(unresolved_sessions.strftime("%Y-%m-%d").tolist())

if len(unexpected_price_dates):
    print("\nUnexpected price dates:")
    print(unexpected_price_dates.strftime("%Y-%m-%d").tolist())

Empty rows on non-trading dates: 53
Expected sessions without complete OHLC: 0
Complete-price rows outside the calendar: 0
Partially missing OHLC rows: 0


In [24]:
print("Empty rows on non-trading dates:", len(removable_dates))
print("Expected sessions without complete OHLC:", len(unresolved_sessions))
print("Complete-price rows outside the calendar:", len(unexpected_price_dates))
print("Partially missing OHLC rows:", partial_prices.sum())

# Show every unresolved date, including dates absent from the download.
unresolved_details = ftse_working.reindex(unresolved_sessions)[
    ["Open", "High", "Low", "Close"]
].copy()

unresolved_details["row_in_download"] = unresolved_sessions.isin(
    observed_dates
)

print("\nUnresolved sessions:")
print(unresolved_details.to_string())

if len(unexpected_price_dates):
    print("\nPrice observations outside the calendar:")
    print(unexpected_price_dates.strftime("%Y-%m-%d").tolist())

Empty rows on non-trading dates: 53
Expected sessions without complete OHLC: 0
Complete-price rows outside the calendar: 0
Partially missing OHLC rows: 0

Unresolved sessions:
Empty DataFrame
Columns: [Open, High, Low, Close, row_in_download]
Index: []


In [25]:
# Stop if any observations remain unresolved.
assert len(unresolved_sessions) == 0
assert len(unexpected_price_dates) == 0
assert partial_prices.sum() == 0
assert len(removable_dates) == 53

# Record the empty non-trading rows before removing them.
remove_mask = observed_dates.isin(removable_dates)

removed_rows = ftse_working.loc[remove_mask].copy()
removed_rows["removal_reason"] = (
    "All OHLC missing; date is not an XLON session"
)
removed_rows["calendar_version"] = xcals.__version__

# Keep trading observations, including both corrected dates.
ftse_clean = ftse_working.loc[~remove_mask].copy()

assert ftse_clean[price_columns].notna().all().all()
assert ftse_clean.index.is_unique
assert ftse_clean.index.is_monotonic_increasing

# Save the cleaned prices and removal log.
processed_folder = project_root / "data" / "processed"
tables_folder = project_root / "outputs" / "tables"

processed_folder.mkdir(parents=True, exist_ok=True)
tables_folder.mkdir(parents=True, exist_ok=True)

ftse_clean.to_csv(
    processed_folder / "ftse100_daily_clean.csv",
    index_label="Date",
)

removed_rows.to_csv(
    tables_folder / "removed_nontrading_rows.csv",
    index_label="Date",
)

print("Clean price observations:", len(ftse_clean))
print("Rows removed:", len(removed_rows))
print("Corrected observations:", ftse_clean["price_corrected"].sum())

Clean price observations: 6756
Rows removed: 53
Corrected observations: 2


In [26]:
import numpy as np

# Work from the cleaned prices.
ftse_returns = ftse_clean.copy()

# Logarithms require positive, finite prices.
assert np.isfinite(ftse_returns["Close"]).all()
assert (ftse_returns["Close"] > 0).all()

# shift(1) takes the previous row: the previous trading session.
ftse_returns["previous_close"] = ftse_returns["Close"].shift(1)

ftse_returns["return_pct"] = 100 * np.log(
    ftse_returns["Close"] / ftse_returns["previous_close"]
)

# A return uses two prices, so flag either one being corrected.
ftse_returns["return_uses_corrected_price"] = (
    ftse_returns["price_corrected"]
    | ftse_returns["price_corrected"].shift(1, fill_value=False)
)

# Trim to the analysis period AFTER calculating returns.
ftse_returns = ftse_returns.loc[
    "2000-01-01":"2026-08-31"
].copy()

assert np.isfinite(ftse_returns["return_pct"]).all()

print("First return date:", ftse_returns.index.min().date())
print("Last return date:", ftse_returns.index.max().date())
print("Return observations:", len(ftse_returns))
print("Missing returns:", ftse_returns["return_pct"].isna().sum())

display(
    ftse_returns[
        ["previous_close", "Close", "return_pct"]
    ].head()
)

First return date: 2000-01-04
Last return date: 2026-08-28
Return observations: 6736
Missing returns: 0


,previous_close,Close,return_pct
Date,,,
2000-01-04,6930.200195,6665.899902,-3.888374
2000-01-05,6665.899902,6535.899902,-1.969492
2000-01-06,6535.899902,6447.200195,-1.366408
2000-01-07,6447.200195,6504.799805,0.889438
2000-01-10,6504.799805,6607.700195,1.569533


In [27]:
import math

first_date = ftse_returns.index[0]
first_row = ftse_returns.iloc[0]

previous_date = ftse_clean.index[
    ftse_clean.index.get_loc(first_date) - 1
]

manual_return = 100 * math.log(
    first_row["Close"] / first_row["previous_close"]
)

print("Previous trading date:", previous_date.date())
print("Return date:", first_date.date())
print(f"Previous close: {first_row['previous_close']:.6f}")
print(f"Current close:  {first_row['Close']:.6f}")
print(f"Manual log return (%): {manual_return:.8f}")
print(f"Stored log return (%): {first_row['return_pct']:.8f}")

assert np.isclose(
    manual_return,
    first_row["return_pct"],
    rtol=0,
    atol=1e-10,
)

Previous trading date: 1999-12-30
Return date: 2000-01-04
Previous close: 6930.200195
Current close:  6665.899902
Manual log return (%): -3.88837374
Stored log return (%): -3.88837374


In [28]:
returns_path = (
    project_root / "data" / "processed" / "ftse100_returns.csv"
)

ftse_returns.to_csv(returns_path, index_label="Date")

print("Saved:", returns_path)

Saved: /Users/lohith/Documents/GitHub/FTSE100-Volatility-Garch/data/processed/ftse100_returns.csv


In [29]:
# Start with the returns dataset.
ftse_model_data = ftse_returns.copy()

# Check the inputs needed for the range-based measure.
range_prices = ftse_model_data[["High", "Low"]]

assert np.isfinite(range_prices.to_numpy()).all()
assert (range_prices > 0).all().all()
assert (
    ftse_model_data["High"] >= ftse_model_data["Low"]
).all()

# Primary variance proxy: squared percentage log return.
ftse_model_data["squared_return"] = (
    ftse_model_data["return_pct"] ** 2
)

# Secondary variance proxy: Parkinson, in matching units.
ftse_model_data["parkinson_var"] = (
    (100 ** 2) / (4 * np.log(2))
    * np.log(
        ftse_model_data["High"] / ftse_model_data["Low"]
    ) ** 2
)

display(
    ftse_model_data[
        ["return_pct", "squared_return", "parkinson_var"]
    ].head()
)

,return_pct,squared_return,parkinson_var
Date,,,
2000-01-04,-3.888374,15.119450,5.580182
2000-01-05,-1.969492,3.878899,2.279758
2000-01-06,-1.366408,1.867072,1.295119
2000-01-07,0.889438,0.791100,0.751246
2000-01-10,1.569533,2.463434,1.399424


In [30]:
proxy_columns = ["squared_return", "parkinson_var"]
proxies = ftse_model_data[proxy_columns]

assert np.isfinite(proxies.to_numpy()).all()
assert (proxies >= 0).all().all()

proxy_checks = pd.DataFrame({
    "valid_observations": proxies.count(),
    "missing_values": proxies.isna().sum(),
    "zero_values": proxies.eq(0).sum(),
})

display(proxy_checks)

print("Dataset rows:", len(ftse_model_data))

,valid_observations,missing_values,zero_values
squared_return,6736,0,8
parkinson_var,6736,0,0


Dataset rows: 6736


In [31]:
modelling_path = (
    project_root
    / "data"
    / "processed"
    / "ftse100_modelling_dataset.csv"
)

ftse_model_data.to_csv(modelling_path, index_label="Date")

print("Saved:", modelling_path)

Saved: /Users/lohith/Documents/GitHub/FTSE100-Volatility-Garch/data/processed/ftse100_modelling_dataset.csv
